# MRIxFields2026 Task 3 — conditional FPS-Former

This notebook only orchestrates the Python implementation in `task3_fpsformer/`. It does not reimplement the model. The workflow is: stage code and the paired prospective data into `/content`, unmount Drive, audit and cache the data, run tests, run a tiny smoke, train a subject-held-out portable pilot, optionally continue on all subjects, infer 180 validation volumes, use the current official builder, validate `task3.zip`, and optionally export selected artifacts.

This copy is configured for one Colab-safe 10-epoch held-out training run: it stages the paired training split, audits and caches it, runs a tiny smoke, trains the portable pilot, and exports the selected checkpoint. Validation inference, all-subject continuation, submission building, and upload remain off.


## 1. Mount Drive, configure paths, and stage the Python package

`DRIVE_DATA_ROOT_VALUE` must be the dataset root containing `training_prospective` or `Training_prospective`. Do not set it to the validation split itself. Put the package zip created on your Mac in Drive, or enable browser upload.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive, files
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

DRIVE_MOUNT = Path('/content/drive')
drive.mount(str(DRIVE_MOUNT))

RUNTIME_ROOT_VALUE = '/content/mrixfields_task3/fpsformer'  # @param {type:'string'}
DRIVE_DATA_ROOT_VALUE = '/content/drive/MyDrive/mrixfields2026'  # @param {type:'string'}
DRIVE_CODE_ZIP_VALUE = '/content/drive/MyDrive/mrixfields_task3_source/task3_fpsformer_colab_fixed.zip'  # @param {type:'string'}
DRIVE_PAPER_PDF_VALUE = ''  # @param {type:'string'}
DRIVE_INITIAL_CHECKPOINT_VALUE = '/content/drive/MyDrive/mrixfields_task3_exports/fpsformer_100/fpsformer_10epoch_01_100/best.pt'  # @param {type:'string'}
DRIVE_EXPORT_ROOT_VALUE = '/content/drive/MyDrive/mrixfields_task3_exports/fpsformer_150_256'  # @param {type:'string'}
USE_BROWSER_UPLOAD_FOR_CODE = False  # @param {type:'boolean'}

RUNTIME_ROOT = Path(RUNTIME_ROOT_VALUE).expanduser().resolve()
PROJECT_DIR = RUNTIME_ROOT / 'project'
DATA_DIR = RUNTIME_ROOT / 'data'
PREPROCESSED_DIR = RUNTIME_ROOT / 'preprocessed' / 'fpsformer'
OUTPUT_DIR = RUNTIME_ROOT / 'outputs'
INFERENCE_DIR = RUNTIME_ROOT / 'inference'
SUBMISSION_DIR = RUNTIME_ROOT / 'submission'
RESULTS_DIR = RUNTIME_ROOT / 'results'
REPORT_DIR = RESULTS_DIR / 'reports'
STAGED_CHECKPOINT_DIR = RUNTIME_ROOT / 'staged_checkpoints'
OFFICIAL_REPO = RUNTIME_ROOT / 'official' / 'MRIxFields2026'
DRIVE_DATA_ROOT = Path(DRIVE_DATA_ROOT_VALUE).expanduser()

drive_root = DRIVE_MOUNT.resolve()
if RUNTIME_ROOT == drive_root or drive_root in RUNTIME_ROOT.parents:
    raise ValueError('RUNTIME_ROOT must be under /content, not mounted Drive.')
for path in (PROJECT_DIR, DATA_DIR, PREPROCESSED_DIR, OUTPUT_DIR, INFERENCE_DIR, SUBMISSION_DIR, RESULTS_DIR, REPORT_DIR, STAGED_CHECKPOINT_DIR, OFFICIAL_REPO.parent):
    path.mkdir(parents=True, exist_ok=True)

RUNTIME_WORK_PATHS = (PROJECT_DIR, DATA_DIR, PREPROCESSED_DIR, OUTPUT_DIR, INFERENCE_DIR, SUBMISSION_DIR, RESULTS_DIR, OFFICIAL_REPO)

def drive_is_mounted():
    return os.path.ismount(DRIVE_MOUNT) or (DRIVE_MOUNT / 'MyDrive').is_dir()

def assert_runtime_only():
    if drive_is_mounted():
        raise RuntimeError('Drive must be unmounted before audits, caching, training, inference, or packaging.')
    for path in RUNTIME_WORK_PATHS:
        resolved = Path(path).expanduser().resolve()
        if resolved == drive_root or drive_root in resolved.parents:
            raise RuntimeError(f'Runtime path points to Drive: {resolved}')

if USE_BROWSER_UPLOAD_FOR_CODE:
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith('.zip')]
    if len(zip_names) != 1:
        raise RuntimeError(f'Upload exactly one package zip; received {list(uploaded)}')
    code_zip_source = Path('/content') / zip_names[0]
else:
    code_zip_source = Path(DRIVE_CODE_ZIP_VALUE).expanduser()
if not code_zip_source.is_file():
    raise FileNotFoundError(f'Package zip not found: {code_zip_source}')
runtime_zip = RUNTIME_ROOT / 'task3_fpsformer_colab.zip'
shutil.copy2(code_zip_source, runtime_zip)
with zipfile.ZipFile(runtime_zip) as archive:
    members = archive.namelist()
    unsafe = [name for name in members if Path(name).is_absolute() or '..' in Path(name).parts]
    if unsafe:
        raise RuntimeError(f'Unsafe zip members: {unsafe[:5]}')
    archive.extractall(PROJECT_DIR)
package_dir = PROJECT_DIR / 'task3_fpsformer'
for required in ('__init__.py', 'data.py', 'model.py', 'train.py', 'infer.py', 'requirements-colab.txt'):
    if not (package_dir / required).is_file():
        raise FileNotFoundError(f'Package zip is missing task3_fpsformer/{required}')
paper_source = Path(DRIVE_PAPER_PDF_VALUE).expanduser() if DRIVE_PAPER_PDF_VALUE.strip() else None
if paper_source is not None:
    if not paper_source.is_file():
        raise FileNotFoundError(paper_source)
    shutil.copy2(paper_source, PROJECT_DIR / '2412.10776v1.pdf')
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
os.environ['TORCH_HOME'] = str(RUNTIME_ROOT / '.cache' / 'torch')
print('Runtime project:', PROJECT_DIR)
print('Package:', package_dir)
print('Dataset root selected for staging:', DRIVE_DATA_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runtime project: /content/mrixfields_task3/fpsformer/project
Package: /content/mrixfields_task3/fpsformer/project/task3_fpsformer
Dataset root selected for staging: /content/drive/MyDrive/mrixfields2026


In [ ]:
drive.mount(str(DRIVE_MOUNT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

DRIVE_DATA_ROOT_VALUE = '/content/drive/MyDrive/mrixfields2026'
DRIVE_DATA_ROOT = Path(DRIVE_DATA_ROOT_VALUE)

training_source = DRIVE_DATA_ROOT / 'training_prospective'

print('Dataset root:', DRIVE_DATA_ROOT)
print('Training split:', training_source)
print('Training split available:', training_source.is_dir())

assert training_source.is_dir(), training_source

Dataset root: /content/drive/MyDrive/mrixfields2026
Training split: /content/drive/MyDrive/mrixfields2026/training_prospective
Training split available: True


## 2. Install dependencies and verify the Colab GPU

Colab already supplies its CUDA-matched PyTorch build; the package requirements do not replace it.


In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(package_dir / 'requirements-colab.txt')])
import nibabel as nib
import numpy as np
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU before continuing.')
properties = torch.cuda.get_device_properties(0)
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU:', properties.name, round(properties.total_memory / 2**30, 1), 'GiB')


PyTorch: 2.11.0+cu128 CUDA: 12.8
GPU: NVIDIA A100-SXM4-40GB 39.5 GiB


## 3. Stage the paired training and validation splits, then unmount Drive

The required paired training split is staged automatically for this 10-epoch run. Validation data and an initialization checkpoint remain off because neither is needed for the held-out pilot. Only the exact Task 3 split tree is copied; the source dataset is never renamed or modified.


In [ ]:
STAGE_TRAINING_DATA = True  # @param {type:'boolean'}
STAGE_VALIDATION_DATA = True  # @param {type:'boolean'}
STAGE_INITIAL_CHECKPOINT = True  # @param {type:'boolean'}

def resolve_source_split(root, aliases, label):
    root = Path(root).expanduser()
    matches = [root / alias for alias in aliases if (root / alias).is_dir()]
    if len(matches) != 1:
        raise RuntimeError(f'{label}: expected exactly one of {aliases} below dataset root {root}; found {matches}. Set DRIVE_DATA_ROOT_VALUE to the parent mrixfields2026 directory, not a split directory.')
    return matches[0]

def stage_nifti_split(source, destination):
    source = Path(source).expanduser().resolve()
    destination = Path(destination).expanduser().resolve()
    source_files = sorted(source.rglob('*.nii.gz'))
    if not source_files:
        raise RuntimeError(f'No NIfTI files in {source}')
    inventory = {path.relative_to(source): path.stat().st_size for path in source_files}
    required_bytes = sum(inventory.values())
    free_bytes = shutil.disk_usage(RUNTIME_ROOT).free
    if free_bytes < required_bytes + 5 * 2**30:
        raise RuntimeError(f'Insufficient runtime disk: need source bytes plus 5 GiB margin ({required_bytes}), free={free_bytes}')
    destination.mkdir(parents=True, exist_ok=True)
    for index, (relative, size) in enumerate(inventory.items(), 1):
        target = destination / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.is_file() and target.stat().st_size == size:
            continue
        temporary = target.with_name(f'.{target.name}.part')
        shutil.copy2(source / relative, temporary)
        if temporary.stat().st_size != size:
            raise RuntimeError(f'Incomplete copy: {temporary}')
        os.replace(temporary, target)
        if index % 10 == 0 or index == len(inventory):
            print(f'Staged {index}/{len(inventory)} files into {destination}')
    runtime_inventory = {path.relative_to(destination): path.stat().st_size for path in destination.rglob('*.nii.gz')}
    if runtime_inventory != inventory:
        raise RuntimeError(f'Runtime inventory differs from source: {destination}')

if STAGE_TRAINING_DATA:
    training_source = resolve_source_split(DRIVE_DATA_ROOT, ('Training_prospective', 'training_prospective'), 'paired training split')
    stage_nifti_split(training_source, DATA_DIR / 'training_prospective')
else:
    print('Training-data staging is OFF.')
if STAGE_VALIDATION_DATA:
    validation_source = resolve_source_split(DRIVE_DATA_ROOT, ('Validating_prospective', 'validation_prospective', 'validating_prospective'), 'validation split')
    stage_nifti_split(validation_source, DATA_DIR / 'validation_prospective')
else:
    print('Validation-data staging is OFF.')
if STAGE_INITIAL_CHECKPOINT:
    checkpoint_source = Path(DRIVE_INITIAL_CHECKPOINT_VALUE).expanduser()
    if not checkpoint_source.is_file():
        raise FileNotFoundError(checkpoint_source)
    staged_initial_checkpoint = STAGED_CHECKPOINT_DIR / checkpoint_source.name
    shutil.copy2(checkpoint_source, staged_initial_checkpoint)
    print('Staged checkpoint:', staged_initial_checkpoint)
subprocess.run(['sync'], check=True)
drive.flush_and_unmount()
assert_runtime_only()
print('Runtime staging complete. Drive is unmounted; all processing is runtime-only.')


Staged checkpoint: /content/mrixfields_task3/fpsformer/staged_checkpoints/best.pt
Runtime staging complete. Drive is unmounted; all processing is runtime-only.


## 4. Run functional package tests

Colab runs the model/data contract tests. Static tests of the notebook file itself run locally before the source ZIP is created, so an older bundled notebook can no longer cause an irrelevant runtime failure here.


In [ ]:
RUN_PACKAGE_TESTS = True  # @param {type:'boolean'}
assert_runtime_only()
if RUN_PACKAGE_TESTS:
    subprocess.check_call([sys.executable, '-m', 'compileall', '-q', 'task3_fpsformer'])
    test_result = subprocess.run([sys.executable, '-m', 'unittest', '-v', 'task3_fpsformer.tests.test_fpsformer'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(test_result.stdout)
    if test_result.returncode:
        raise RuntimeError(f'Functional package tests failed with exit code {test_result.returncode}. The complete test output is printed above.')
else:
    print('Package tests are OFF.')


test_cache_name_round_trip (task3_fpsformer.tests.test_fpsformer.CommonContractTests.test_cache_name_round_trip) ... ok
test_directed_pairs_have_no_identity_and_cover_twenty (task3_fpsformer.tests.test_fpsformer.CommonContractTests.test_directed_pairs_have_no_identity_and_cover_twenty) ... ok
test_joint_domains_cover_exactly_fifteen_indices (task3_fpsformer.tests.test_fpsformer.CommonContractTests.test_joint_domains_cover_exactly_fifteen_indices) ... ok
test_prediction_path_matches_official_builder_source_layout (task3_fpsformer.tests.test_fpsformer.CommonContractTests.test_prediction_path_matches_official_builder_source_layout) ... ok
test_complete_subject_requires_all_fifteen_domains (task3_fpsformer.tests.test_fpsformer.DataTests.test_complete_subject_requires_all_fifteen_domains) ... ok
test_misaligned_slice_is_rejected (task3_fpsformer.tests.test_fpsformer.DataTests.test_misaligned_slice_is_rejected) ... ok
test_paired_random_crop_uses_identical_coordinates (task3_fpsformer.tests.

## 5. Audit the paired prospective training volumes

This must pass before cache creation. It checks all 15 domains, exact prospective subject pairing, full geometry, float32 dtype, finite values, and `[0,1]` range.


In [ ]:
RUN_RAW_AUDIT = True  # @param {type:'boolean'}
assert_runtime_only()
if RUN_RAW_AUDIT:
    subprocess.check_call([sys.executable, '-m', 'task3_fpsformer.data', 'audit-raw', '--data-dir', str(DATA_DIR), '--check-values', '--report', str(REPORT_DIR / 'raw_audit.json')])
else:
    print('Raw training-data audit is OFF.')


## 6. Build and audit the 220-slice paired cache

Cache creation writes only under runtime storage. It extracts axial indices `[72,292)` and is resumable.


In [ ]:
RUN_CACHE_PREPARATION = True  # @param {type:'boolean'}
assert_runtime_only()
if RUN_CACHE_PREPARATION:
    if not (REPORT_DIR / 'raw_audit.json').is_file():
        raise RuntimeError('Run and pass the raw audit first.')
    subprocess.check_call([sys.executable, '-m', 'task3_fpsformer.data', 'prepare-cache', '--data-dir', str(DATA_DIR), '--cache-dir', str(PREPROCESSED_DIR)])
else:
    print('Cache preparation is OFF.')

RUN_CACHE_AUDIT = True  # @param {type:'boolean'}
if RUN_CACHE_AUDIT:
    subprocess.check_call([sys.executable, '-m', 'task3_fpsformer.data', 'audit-cache', '--cache-dir', str(PREPROCESSED_DIR), '--report', str(REPORT_DIR / 'cache_audit.json')])
else:
    print('Cache audit is OFF.')


## 7. Two-update tiny smoke training

This tests the complete data/model/loss/checkpoint/holdout path. It is not a quality result. Set the baseline acknowledgement only because your organizer-pretrained 180-case milestone has already completed. The smoke deliberately uses `--workers 0` so a Colab multiprocessing/shared-memory failure cannot hide the model result.


In [ ]:
BASELINE_MILESTONE_COMPLETE = True  # Verified locally: 180/180 baseline cases and validated task3.zip
RUN_TINY_SMOKE = False  # @param {type:'boolean'}
TINY_SMOKE_DIR = OUTPUT_DIR / 'tiny_smoke'
assert_runtime_only()
if RUN_TINY_SMOKE:
    if not BASELINE_MILESTONE_COMPLETE:
        raise RuntimeError('Acknowledge the completed baseline milestone before training.')
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is no longer available; reconnect to a Colab GPU runtime.')
    cache_audit_path = REPORT_DIR / 'cache_audit.json'
    if not cache_audit_path.is_file():
        raise RuntimeError('Run and pass the cache audit first.')
    cache_audit = json.loads(cache_audit_path.read_text())
    complete_subjects = cache_audit.get('complete_subjects', [])
    if cache_audit.get('status') != 'passed' or len(complete_subjects) < 2:
        raise RuntimeError(f'Tiny holdout smoke requires a passed cache and at least two complete subjects; status={cache_audit.get("status")}, subjects={complete_subjects}')
    existing_smoke_checkpoints = sorted((TINY_SMOKE_DIR / 'checkpoints').glob('*.pt'))
    if existing_smoke_checkpoints:
        raise RuntimeError(f'Tiny smoke already has checkpoints; it passed previously or needs an explicit fresh output directory: {existing_smoke_checkpoints}')
    print('Tiny smoke preflight passed. Complete subjects:', complete_subjects)
    subprocess.check_call([sys.executable, '-m', 'task3_fpsformer.train', '--cache-dir', str(PREPROCESSED_DIR), '--output-dir', str(TINY_SMOKE_DIR), '--preset', 'tiny', '--device', 'cuda', '--amp', 'fp16', '--crop-size', '256', '--eval-crop-size', '0', '--epochs', '1', '--updates-per-epoch', '2', '--val-samples-per-pair', '1', '--workers', '0', '--progress', '--baseline-milestone-complete'])
else:
    print('Tiny smoke training is OFF.')


Tiny smoke training is OFF.


## 8. Subject-held-out portable pilot

This run uses 10 epochs with 100 optimizer updates each (1,000 total), 128 x 128 crops, fp16, and no DataLoader worker processes. A rate-limited progress bar shows every training epoch and holdout pass without flooding the browser. Metrics are balanced slice proxies on one untouched prospective training subject, not public-validation scores.


In [ ]:
checkpoint_path = next(STAGED_CHECKPOINT_DIR.glob("*.pt"))
checkpoint = torch.load(checkpoint_path, map_location="cpu",
weights_only=True)

print("epoch:", checkpoint["epoch"])
print("global_step:", checkpoint["global_step"])
print(checkpoint_path)

epoch: 53
global_step: 5300
/content/mrixfields_task3/fpsformer/staged_checkpoints/best.pt


In [ ]:
RUN_PORTABLE_PILOT = True  # @param {type:'boolean'}
RESUME_PORTABLE_PILOT = True  # @param {type:'boolean'}
PILOT_RESUME_CHECKPOINT_VALUE = "/content/mrixfields_task3/fpsformer/staged_checkpoints/best.pt"  # @param {type:'string'}
PILOT_EPOCHS = 150  # @param {type:'integer'}
PILOT_UPDATES_PER_EPOCH = 100  # @param {type:'integer'}
EVAL_LPIPS = True  # @param {type:'boolean'}
PILOT_DIR = OUTPUT_DIR / f"portable_pilot_epoch_10_to_{PILOT_EPOCHS}"
assert_runtime_only()
if RUN_PORTABLE_PILOT:
    if not BASELINE_MILESTONE_COMPLETE:
        raise RuntimeError('Acknowledge the completed baseline milestone before training.')
    command = [sys.executable, '-m', 'task3_fpsformer.train', '--cache-dir', str(PREPROCESSED_DIR), '--output-dir', str(PILOT_DIR), '--preset', 'portable', '--device', 'cuda', '--amp', 'fp16', '--batch-size', '16', '--gradient-accumulation', '1', '--crop-size', '256', '--eval-crop-size', '0', '--epochs', str(PILOT_EPOCHS), '--updates-per-epoch', str(PILOT_UPDATES_PER_EPOCH), '--val-samples-per-pair', '1', '--workers', '0', '--progress', '--baseline-milestone-complete']
    if RESUME_PORTABLE_PILOT:
        resume_candidates = [Path(PILOT_RESUME_CHECKPOINT_VALUE).expanduser()] if PILOT_RESUME_CHECKPOINT_VALUE.strip() else sorted(STAGED_CHECKPOINT_DIR.glob('*.pt')) + sorted(STAGED_CHECKPOINT_DIR.glob('*.pth'))
        if len(resume_candidates) != 1 or not resume_candidates[0].is_file():
            raise RuntimeError(f'Resume requires exactly one staged or explicit checkpoint; found {resume_candidates}')
        resume_checkpoint = resume_candidates[0]
        checkpoint_metadata = torch.load(resume_checkpoint, map_location='cpu', weights_only=True)
        print('Resume checkpoint:', resume_checkpoint)
        print('Resume epoch:', checkpoint_metadata.get('epoch'))
        print('Resume global step:', checkpoint_metadata.get('global_step'))
        print('Resume holdout:', checkpoint_metadata.get('holdout_subject'))
        print('Resume model config:', checkpoint_metadata.get('model_config'))
        command.extend(['--resume', str(resume_checkpoint)])
    if EVAL_LPIPS:
        command.append('--eval-lpips')
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    output_tail = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        output_tail.append(line)
        output_tail = output_tail[-100:]
    return_code = process.wait()
    if return_code:
        raise RuntimeError(f'Training failed with exit code {return_code}. Last child output:\n{"".join(output_tail)}')
else:
    print('Portable pilot training is OFF.')


## 9. Review training loss and holdout metrics

For nRMSE and LPIPS lower is better; for SSIM higher is better. Do not select on public validation repeatedly before the prospective holdout experiment is understood.


In [ ]:
assert_runtime_only()
metrics_path = PILOT_DIR / 'metrics.jsonl'
if metrics_path.is_file():
    metric_records = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
    loss_records = [record for record in metric_records if record.get('kind') == 'epoch' and record.get('train')]
    for record in loss_records:
        print('epoch', record['epoch'], 'step', record['global_step'], 'train loss', json.dumps(record['train'], indent=2), 'holdout', json.dumps(record.get('holdout', {}), indent=2))
    if loss_records:
        import matplotlib.pyplot as plt
        loss_epochs = [record['epoch'] for record in loss_records]
        total_losses = [record['train']['total'] for record in loss_records]
        plt.figure(figsize=(8, 4))
        plt.plot(loss_epochs, total_losses, marker='o', label='weighted total')
        plt.xlabel('Epoch')
        plt.ylabel('Training loss')
        plt.title('Task 3 FPS-Former training loss')
        plt.grid(alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.show()
else:
    print('Run the portable pilot first:', metrics_path)


epoch 54 step 5400 {
  "by_modality": {
    "T1W": {
      "samples": 20,
      "slice_proxy_lpips": 0.12381688244640827,
      "slice_proxy_nrmse": 0.30775787504995844,
      "slice_proxy_ssim": 0.9217944145202637
    },
    "T2FLAIR": {
      "samples": 20,
      "slice_proxy_lpips": 0.08941790033131838,
      "slice_proxy_nrmse": 0.19938699548423783,
      "slice_proxy_ssim": 0.9374252945184708
    },
    "T2W": {
      "samples": 20,
      "slice_proxy_lpips": 0.11146140545606613,
      "slice_proxy_nrmse": 0.22346397071699767,
      "slice_proxy_ssim": 0.9291594833135605
    }
  },
  "overall": {
    "samples": 60,
    "slice_proxy_lpips": 0.1082320627445976,
    "slice_proxy_nrmse": 0.24704969340570385,
    "slice_proxy_ssim": 0.9294597307840983
  }
}
epoch 55 step 5500 {
  "by_modality": {
    "T1W": {
      "samples": 20,
      "slice_proxy_lpips": 0.12348685832694173,
      "slice_proxy_nrmse": 0.33746620938608435,
      "slice_proxy_ssim": 0.9181938201189042
    },
    "T2FLA

## 10. Optional all-subject continuation

After selecting settings on the held-out pilot, initialize a fresh run with model weights only and include all paired prospective training subjects. This run has no honest local holdout.


In [ ]:
RUN_ALL_SUBJECT_CONTINUATION = False  # @param {type:'boolean'}
ALL_SUBJECT_EPOCHS = 10  # @param {type:'integer'}
ALL_SUBJECT_UPDATES_PER_EPOCH = 100  # @param {type:'integer'}
ALL_SUBJECT_DIR = OUTPUT_DIR / 'all_subjects'
assert_runtime_only()
if RUN_ALL_SUBJECT_CONTINUATION:
    if not BASELINE_MILESTONE_COMPLETE:
        raise RuntimeError('Acknowledge the completed baseline milestone before training.')
    pilot_checkpoint = PILOT_DIR / 'checkpoints' / 'best.pt'
    if not pilot_checkpoint.is_file():
        raise FileNotFoundError(pilot_checkpoint)
    subprocess.check_call([sys.executable, '-m', 'task3_fpsformer.train', '--cache-dir', str(PREPROCESSED_DIR), '--output-dir', str(ALL_SUBJECT_DIR), '--preset', 'portable', '--initialize', str(pilot_checkpoint), '--holdout-subject', 'none', '--device', 'cuda', '--amp', 'fp16', '--batch-size', '1', '--gradient-accumulation', '1', '--crop-size', '128', '--epochs', str(ALL_SUBJECT_EPOCHS), '--updates-per-epoch', str(ALL_SUBJECT_UPDATES_PER_EPOCH), '--workers', '0', '--progress', '--baseline-milestone-complete'])
else:
    print('All-subject continuation is OFF.')


All-subject continuation is OFF.


## 11. Select one checkpoint and run full-volume validation smoke

Leave the checkpoint field blank to prefer the all-subject checkpoint, then the pilot, then any checkpoint staged in step 3. The smoke generates exactly one complete 364-slice prediction and validates its metadata and values.


In [ ]:
new_best = PILOT_DIR / "checkpoints" / "best.pt"
original_checkpoint = Path(PILOT_RESUME_CHECKPOINT_VALUE)

SELECTED_CHECKPOINT = (
    new_best if new_best.is_file() else original_checkpoint
)

selected = torch.load(
    SELECTED_CHECKPOINT,
    map_location="cpu",
    weights_only=True,
)

print("Selected epoch:", selected["epoch"])
print("Best nRMSE:", selected["best_nrmse"])

Selected epoch: 95
Best nRMSE: 0.21299595057918827


In [ ]:
best_path = PILOT_DIR / "checkpoints" / "best.pt"
checkpoint = torch.load(best_path, map_location="cpu", weights_only=True)

print("epoch:", checkpoint["epoch"])
print("best nRMSE:", checkpoint["best_nrmse"])

assert checkpoint["epoch"] == 95
SELECTED_CHECKPOINT_VALUE = str(best_path)


epoch: 95
best nRMSE: 0.21299595057918827


In [ ]:
import torch
from task3_fpsformer.infer import inventory_inputs

checkpoint = torch.load(
    SELECTED_CHECKPOINT,
    map_location="cpu",
    weights_only=True,
)

print("Checkpoint epoch:", checkpoint["epoch"])
print("Checkpoint format:", checkpoint.get("format"))
print("Validation files:", len(list(DATA_DIR.rglob("*.nii.gz"))))
print("Audited validation volumes:", len(inventory_inputs(DATA_DIR)))

Checkpoint epoch: 95
Checkpoint format: task3_conditional_fpsformer_v1
Validation files: 90
Audited validation volumes: 45


In [ ]:
from task3_fpsformer.infer import inventory_inputs

validation_root = DATA_DIR / "validation_prospective"
print("Files:", len(list(validation_root.rglob("*.nii.gz"))))
print("Audited volumes:", len(inventory_inputs(DATA_DIR)))

Files: 45
Audited volumes: 45


In [ ]:
import subprocess
import sys
import time

debug_dir = INFERENCE_DIR / f"smoke_epoch95_{time.time_ns()}"

command = [
    sys.executable, "-m", "task3_fpsformer.infer",
    "--checkpoint", str(SELECTED_CHECKPOINT),
    "--input-root", str(DATA_DIR),
    "--output-dir", str(debug_dir),
    "--device", "cuda",
    "--amp", "fp16",
    "--batch-size", "1",
    "--smoke",
]

result = subprocess.run(
    command,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(result.stdout)
print("Return code:", result.returncode)
print("Output directory:", debug_dir)

[001/001] T1W 0.1T->1.5T P_0001: OK
Inference complete: 1 validated volumes; manifest=/content/mrixfields_task3/fpsformer/inference/smoke_epoch95_1788172325741835655/task3_fpsformer_manifest.json

Return code: 0
Output directory: /content/mrixfields_task3/fpsformer/inference/smoke_epoch95_1788172325741835655


In [ ]:
infer_path = PROJECT_DIR / "task3_fpsformer" / "infer.py"
source = infer_path.read_text()

old = "torch.from_numpy(moved[start:stop])"
new = "torch.from_numpy(np.ascontiguousarray(moved[start:stop]))"

if old in source:
    infer_path.write_text(source.replace(old, new, 1))
    print("Patched:", infer_path)
elif new in source:
    print("Patch already applied:", infer_path)
else:
    raise RuntimeError("Expected inference line was not found")

compile(infer_path.read_text(), str(infer_path), "exec")
print("Syntax check passed")

Patch already applied: /content/mrixfields_task3/fpsformer/project/task3_fpsformer/infer.py
Syntax check passed


In [ ]:
VALIDATION_SMOKE_DIR = INFERENCE_DIR / "smoke_epoch14_fixed"

In [ ]:
SELECTED_CHECKPOINT_VALUE = '/content/mrixfields_task3/fpsformer/outputs/portable_pilot_epoch_10_to_150/checkpoints/best.pt'  # @param {type:'string'}
RUN_VALIDATION_SMOKE = True  # @param {type:'boolean'}
VALIDATION_SMOKE_DIR = INFERENCE_DIR / "smoke_epoch95"
assert_runtime_only()
staged_candidates = sorted(STAGED_CHECKPOINT_DIR.glob('*.pt')) + sorted(STAGED_CHECKPOINT_DIR.glob('*.pth'))
checkpoint_candidates = [Path(SELECTED_CHECKPOINT_VALUE).expanduser()] if SELECTED_CHECKPOINT_VALUE.strip() else [ALL_SUBJECT_DIR / 'checkpoints' / 'best.pt', PILOT_DIR / 'checkpoints' / 'best.pt', *staged_candidates]
SELECTED_CHECKPOINT = next((path for path in checkpoint_candidates if path.is_file()), checkpoint_candidates[0])
print('Selected checkpoint:', SELECTED_CHECKPOINT)
if RUN_VALIDATION_SMOKE:
    if not SELECTED_CHECKPOINT.is_file():
        raise FileNotFoundError(SELECTED_CHECKPOINT)
    subprocess.check_call([sys.executable, '-m', 'task3_fpsformer.infer', '--checkpoint', str(SELECTED_CHECKPOINT), '--input-root', str(DATA_DIR), '--output-dir', str(VALIDATION_SMOKE_DIR), '--device', 'cuda', '--amp', 'fp16', '--batch-size', '1', '--smoke'])
else:
    print('Validation smoke inference is OFF.')


Selected checkpoint: /content/mrixfields_task3/fpsformer/outputs/portable_pilot_epoch_10_to_150/checkpoints/best.pt


## 12. Complete resumable 180-volume validation sweep

Public validation targets are private. This validates the prediction structure and numerical properties but does not produce a local leaderboard metric.


In [ ]:
RUN_FULL_VALIDATION = True  # @param {type:'boolean'}
FULL_INFERENCE_DIR = INFERENCE_DIR / "full_epoch14"
assert_runtime_only()
assert "np.ascontiguousarray(moved[start:stop])" in (
    PROJECT_DIR / "task3_fpsformer" / "infer.py"
).read_text()

if RUN_FULL_VALIDATION:
    if not SELECTED_CHECKPOINT.is_file():
        raise FileNotFoundError(SELECTED_CHECKPOINT)

    subprocess.check_call([
        sys.executable, "-m", "task3_fpsformer.infer",
        "--checkpoint", str(SELECTED_CHECKPOINT),
        "--input-root", str(DATA_DIR),
        "--output-dir", str(FULL_INFERENCE_DIR),
        "--device", "cuda",
        "--amp", "fp16",
        "--batch-size", "16",
        "--resume",
    ])
else:
    print("Full validation inference is OFF.")


## 13. Fetch the current official builder and review live rules

Immediately before packaging, open the two URLs printed below. The notebook records the current organizer Git commit instead of silently using the old prepared pin.


In [ ]:
PREPARE_OFFICIAL_REPO = True  # @param {type:'boolean'}
LIVE_RULES_REVIEWED = True  # @param {type:'boolean'}
LIVE_RULE_URLS = ('https://mrixfields.chihucloud.com/2026/tasks.html', 'https://github.com/MRIxFields/MRIxFields2026/blob/main/Submission/README.md')
assert_runtime_only()
print('Review before packaging:', *LIVE_RULE_URLS, sep='\n - ')
if PREPARE_OFFICIAL_REPO:
    if OFFICIAL_REPO.exists() and not (OFFICIAL_REPO / '.git').is_dir():
        raise RuntimeError(f'Official repo destination exists but is not a Git checkout: {OFFICIAL_REPO}')
    if not (OFFICIAL_REPO / '.git').is_dir():
        subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/MRIxFields/MRIxFields2026.git', str(OFFICIAL_REPO)])
    official_revision = subprocess.check_output(['git', '-C', str(OFFICIAL_REPO), 'rev-parse', 'HEAD'], text=True).strip()
    submission_readme = OFFICIAL_REPO / 'Submission' / 'README.md'
    official_report = {'created_at_utc': __import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat(), 'official_revision': official_revision, 'submission_readme_sha256': hashlib.sha256(submission_readme.read_bytes()).hexdigest(), 'live_rule_urls': list(LIVE_RULE_URLS), 'rules_reviewed': LIVE_RULES_REVIEWED}
    (REPORT_DIR / 'official_builder_provenance.json').write_text(json.dumps(official_report, indent=2) + '\n')
    print(json.dumps(official_report, indent=2))
else:
    print('Official builder preparation is OFF.')


Review before packaging:
 - https://mrixfields.chihucloud.com/2026/tasks.html
 - https://github.com/MRIxFields/MRIxFields2026/blob/main/Submission/README.md
{
  "created_at_utc": "2026-08-31T11:54:02.671442+00:00",
  "official_revision": "a22abccbbf61e2131d27331035ff5b9114565fbd",
  "submission_readme_sha256": "aa3bbb749dfc0b5d347133e08d0640f7907996e663b695338f44054105f703d0",
  "live_rule_urls": [
    "https://mrixfields.chihucloud.com/2026/tasks.html",
    "https://github.com/MRIxFields/MRIxFields2026/blob/main/Submission/README.md"
  ],
  "rules_reviewed": true
}


## 14. Build, validate, and zip the experimental Task 3 submission

This calls the current official builder with method `fpsformer`, then checks 60 pair directories, 180 files, no identity mappings, no segmentation content, correct target-field filenames, `(364,436,30)` float32 volumes, finite `[0,1]` values, and a top-level `task3/` archive. It never uploads the archive.


In [ ]:
BUILD_TASK3_ZIP = True  # @param {type:'boolean'}
assert_runtime_only()
if BUILD_TASK3_ZIP:
    if not LIVE_RULES_REVIEWED:
        raise RuntimeError('Review the live Task 3 and submission rules first.')
    builder = OFFICIAL_REPO / 'Submission' / 'build_submission' / 'build_submission.py'
    if not builder.is_file():
        raise FileNotFoundError('Prepare the current official repository first.')
    manifest_path = FULL_INFERENCE_DIR / 'task3_fpsformer_manifest.json'
    manifest = json.loads(manifest_path.read_text())
    if manifest.get('status') != 'complete' or manifest.get('planned_cases') != 180 or len(manifest.get('completed_cases', [])) != 180:
        raise RuntimeError('Full inference manifest is not a complete 180-case run.')
    submission_run_dir = SUBMISSION_DIR / 'fpsformer_best'
    if submission_run_dir.exists():
        raise RuntimeError(f'Use a fresh submission output directory: {submission_run_dir}')
    placeholder = RUNTIME_ROOT / 'unused_task3_seg_placeholder'
    placeholder.mkdir(parents=True, exist_ok=True)
    environment = os.environ.copy()
    environment['PYTHONPATH'] = str(OFFICIAL_REPO / 'Baseline') + os.pathsep + environment.get('PYTHONPATH', '')
    subprocess.check_call([sys.executable, str(builder), '--tasks', 'task3', '--predictions-dir', str(FULL_INFERENCE_DIR), '--predictions-seg-dir', str(placeholder), '--output-dir', str(submission_run_dir), '--task3-method', 'fpsformer', '--task3-mode', 'pro_scratch', '--task3-epoch', 'best'], env=environment)
    task_root = submission_run_dir / 'task3'
    modalities = ('T1W', 'T2W', 'T2FLAIR')
    fields = ('0.1T', '1.5T', '3T', '5T', '7T')
    expected_ids = {'0.1T': ('0001', '0002', '0003'), '1.5T': ('0004', '0005', '0008'), '3T': ('0010', '0011', '0012'), '5T': ('0013', '0014', '0015'), '7T': ('0016', '0017', '0018')}
    expected_files = set()
    expected_pred_dirs = set()
    for modality in modalities:
        for source_field in fields:
            for target_field in fields:
                if source_field == target_field:
                    continue
                pred_dir = task_root / modality / f'{source_field}_to_{target_field}' / 'pred'
                expected_pred_dirs.add(pred_dir)
                for subject_id in expected_ids[source_field]:
                    expected_files.add(pred_dir / f'P_{modality}_{target_field}_{subject_id}.nii.gz')
    actual_files = set(task_root.rglob('*.nii.gz'))
    actual_pred_dirs = {path for path in task_root.rglob('pred') if path.is_dir()}
    if actual_files != expected_files or actual_pred_dirs != expected_pred_dirs:
        raise RuntimeError(f'Submission tree mismatch: files={len(actual_files)}/180, pred_dirs={len(actual_pred_dirs)}/60')
    if any('seg' in path.parts or path.name.endswith('_seg.nii.gz') for path in task_root.rglob('*')):
        raise RuntimeError('Task 3 submission contains segmentation content.')
    numerical = []
    for index, path in enumerate(sorted(actual_files), 1):
        image = nib.load(str(path))
        data = image.get_fdata(dtype=np.float32)
        if tuple(image.shape) != (364, 436, 30) or str(image.get_data_dtype()) != 'float32' or not np.isfinite(data).all() or data.min() < -1e-6 or data.max() > 1 + 1e-6:
            raise RuntimeError(f'Invalid slab prediction: {path}')
        numerical.append({'path': str(path.relative_to(submission_run_dir)), 'minimum': float(data.min()), 'maximum': float(data.max())})
        if index % 20 == 0 or index == len(actual_files):
            print(f'Validated {index}/{len(actual_files)} slab predictions')
    archive_path = RESULTS_DIR / 'task3.zip'
    if archive_path.exists():
        raise RuntimeError(f'Archive already exists; preserve it and use a fresh runtime path: {archive_path}')
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        archive.write(task_root, 'task3/')
        for path in sorted(task_root.rglob('*')):
            archive.write(path, path.relative_to(submission_run_dir))
    with zipfile.ZipFile(archive_path) as archive:
        names = archive.namelist()
        if not names or not all(name == 'task3/' or name.startswith('task3/') for name in names):
            raise RuntimeError('Archive does not have exactly one top-level task3/ directory.')
    digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
    build_report = {'status': 'ready_for_user_upload', 'archive': str(archive_path), 'archive_size_bytes': archive_path.stat().st_size, 'archive_sha256': digest, 'official_revision': subprocess.check_output(['git', '-C', str(OFFICIAL_REPO), 'rev-parse', 'HEAD'], text=True).strip(), 'checkpoint_sha256': manifest['checkpoint_sha256'], 'prediction_files': len(actual_files), 'pair_directories': len(actual_pred_dirs), 'numerical_checks': numerical}
    (REPORT_DIR / 'task3_fpsformer_submission_report.json').write_text(json.dumps(build_report, indent=2) + '\n')
    print(json.dumps({key: value for key, value in build_report.items() if key != 'numerical_checks'}, indent=2))
else:
    print('Task 3 build is OFF.')


Validated 20/180 slab predictions
Validated 40/180 slab predictions
Validated 60/180 slab predictions
Validated 80/180 slab predictions
Validated 100/180 slab predictions
Validated 120/180 slab predictions
Validated 140/180 slab predictions
Validated 160/180 slab predictions
Validated 180/180 slab predictions
{
  "status": "ready_for_user_upload",
  "archive": "/content/mrixfields_task3/fpsformer/results/task3.zip",
  "archive_size_bytes": 1430526774,
  "archive_sha256": "cbe53ee890b59cefb43145cb4d971f03d937d5510a64ee1a48c88edeb79f1e75",
  "official_revision": "a22abccbbf61e2131d27331035ff5b9114565fbd",
  "checkpoint_sha256": "eebd706518d30601d78f8b2aa5ea586d9f9deb9e55147a60b63b61f75545e98f",
  "prediction_files": 180,
  "pair_directories": 60
}


## 15. Export only selected final artifacts

This is the only post-staging cell that remounts Drive. It copies one selected checkpoint plus compact reports/metrics and optionally the validated archive, verifies every copy, and unmounts Drive again. It does not export caches or the full prediction tree.


In [ ]:
EXPORT_SELECTED_ARTIFACTS = True  # @param {type:'boolean'}
EXPORT_NAME_VALUE = 'fpsformer_10epoch_01_150_16'  # @param {type:'string'}
EXPORT_CHECKPOINT = True  # @param {type:'boolean'}
EXPORT_ARCHIVE = True  # @param {type:'boolean'}
if EXPORT_SELECTED_ARTIFACTS:
    if drive_is_mounted():
        raise RuntimeError('Drive unexpectedly mounted before export.')
    drive.mount(str(DRIVE_MOUNT))
    try:
        export_root = Path(DRIVE_EXPORT_ROOT_VALUE).expanduser() / EXPORT_NAME_VALUE
        if export_root.exists():
            raise RuntimeError(f'Export destination exists; choose a new name: {export_root}')
        export_root.mkdir(parents=True)
        sources = []
        selected = globals().get('SELECTED_CHECKPOINT')
        if EXPORT_CHECKPOINT:
            if selected is None or not Path(selected).is_file():
                raise FileNotFoundError(f'Selected checkpoint is unavailable: {selected}')
            sources.append(Path(selected))
        if EXPORT_ARCHIVE:
            archive_source = RESULTS_DIR / 'task3.zip'
            if not archive_source.is_file():
                raise FileNotFoundError(archive_source)
            sources.append(archive_source)
        for directory in (PILOT_DIR, ALL_SUBJECT_DIR, REPORT_DIR):
            if directory.is_dir():
                sources.extend(path for path in directory.rglob('*') if path.is_file() and (path.name in {'metrics.jsonl', 'provenance.json'} or path.suffix == '.json'))
        seen = set()
        export_records = []
        for source in sources:
            source = source.resolve()
            if source in seen:
                continue
            seen.add(source)
            destination = export_root / source.name
            if destination.exists():
                destination = export_root / f'{source.parent.name}_{source.name}'
            shutil.copy2(source, destination)
            source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
            destination_hash = hashlib.sha256(destination.read_bytes()).hexdigest()
            if source.stat().st_size != destination.stat().st_size or source_hash != destination_hash:
                raise RuntimeError(f'Export verification failed: {destination}')
            export_records.append({'source': str(source), 'destination': str(destination), 'size': source.stat().st_size, 'sha256': source_hash})
        (export_root / 'export_manifest.json').write_text(json.dumps(export_records, indent=2) + '\n')
        print('Verified exports:', json.dumps(export_records, indent=2))
    finally:
        drive.flush_and_unmount()
else:
    print('Artifact export is OFF. Runtime results remain under', RUNTIME_ROOT)


Mounted at /content/drive
Verified exports: [
  {
    "source": "/content/mrixfields_task3/fpsformer/outputs/portable_pilot_epoch_10_to_150/checkpoints/best.pt",
    "destination": "/content/drive/MyDrive/mrixfields_task3_exports/fpsformer_150_256/fpsformer_10epoch_01_150_16/best.pt",
    "size": 138870669,
    "sha256": "eebd706518d30601d78f8b2aa5ea586d9f9deb9e55147a60b63b61f75545e98f"
  },
  {
    "source": "/content/mrixfields_task3/fpsformer/results/task3.zip",
    "destination": "/content/drive/MyDrive/mrixfields_task3_exports/fpsformer_150_256/fpsformer_10epoch_01_150_16/task3.zip",
    "size": 1430526774,
    "sha256": "cbe53ee890b59cefb43145cb4d971f03d937d5510a64ee1a48c88edeb79f1e75"
  },
  {
    "source": "/content/mrixfields_task3/fpsformer/outputs/portable_pilot_epoch_10_to_150/metrics.jsonl",
    "destination": "/content/drive/MyDrive/mrixfields_task3_exports/fpsformer_150_256/fpsformer_10epoch_01_150_16/metrics.jsonl",
    "size": 91431,
    "sha256": "61c19a7cec565cfc2e7c

## COMPARE

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

export_root = Path(
    "/content/drive/MyDrive/mrixfields_task3_exports"
)

for path in sorted(export_root.rglob("*")):
    if path.is_file() and (
        path.suffix in {".pt", ".zip"}
        or path.name in {"metrics.jsonl", "provenance.json"}
    ):
        print(path)


In [ ]:
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np

original = nib.load("original_path").get_fdata(dtype=np.float32)
reconstructed = nib.load("reconstructed_path").get_fdata(dtype=np.float32)

assert original.shape == (364, 436, 364)
assert reconstructed.shape == (364, 436, 30)

images = [
    ("Original source: 0.1T", original[:, :, 165]),
    ("Reconstructed target: 7T", reconstructed[:, :, 15]),
    ("Absolute difference", np.abs(original[:, :, 165] - reconstructed[:, :,
    15])),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (title, image) in zip(axes, images):
    ax.imshow(image.T, cmap="gray", origin="lower", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
for path in OUTPUT_DIR.rglob("checkpoints/*.pt"):
    state = torch.load(path, map_location="cpu", weights_only=True)
    print(path, "epoch:", state["epoch"], "step:", state["global_step"])

from google.colab import drive
import hashlib
import shutil

drive.mount(str(DRIVE_MOUNT))

export_root = (
    Path(DRIVE_EXPORT_ROOT_VALUE)
    / "fpsformer_10epoch_01"  # use your actual existing folder
)

last_checkpoint = PILOT_DIR / "checkpoints" / "last.pt"
inference_manifest = (
    FULL_INFERENCE_DIR / "task3_fpsformer_manifest.json"
)

for source in (last_checkpoint, inference_manifest):
    if not source.is_file():
        raise FileNotFoundError(source)

    destination = export_root / source.name
    shutil.copy2(source, destination)

    source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
    destination_hash = hashlib.sha256(destination.read_bytes()).hexdigest()
    assert source_hash == destination_hash

    print("Saved:", destination)
    print("SHA-256:", source_hash)

drive.flush_and_unmount()

/content/mrixfields_task3/fpsformer/outputs/portable_pilot_epoch_10_to_30/checkpoints/best.pt epoch: 13 step: 1300
/content/mrixfields_task3/fpsformer/outputs/portable_pilot_epoch_10_to_30/checkpoints/last.pt epoch: 30 step: 3000
Mounted at /content/drive
Saved: /content/drive/MyDrive/mrixfields_task3_exports/fpsformer/fpsformer_10epoch_01/last.pt
SHA-256: 5b497b38333f322f6fb6e929e13b5032e03a962907c9e275674a21192a3af4aa
Saved: /content/drive/MyDrive/mrixfields_task3_exports/fpsformer/fpsformer_10epoch_01/task3_fpsformer_manifest.json
SHA-256: b0d91e18362175b5bc3b7821e622f4f9abc9ac61b7b8b6973d9dce7c55e8b15d
